# Mobilscan - 3D Gaussian Splatting Generátor 🚀
Ezzel a programmal az iPhone-oddal szkennelt adatokból valósághű 3D modellt készíthetsz!

**Lépések:**
1. Tömörítsd be az iPhone-odról lementett mappát egy **`Scan.zip`** fájlba (úgy, hogy a zipben egyből az `images` mappa és a `transforms.json` legyen benne).
2. Bal oldalon a **Mappa ikonra 📁** kattintva húzd be a `Scan.zip` fájlt a Colab fájljai közé.
3. Futtasd le sorban az alábbi kódblokkokat (a kis lejátszás gombra kattintva rajtuk)!

### 1. Rendszer előkészítése és NerfStudio telepítése (kb. 3-5 perc)

In [ ]:
!pip install --upgrade pip
!pip install nerfstudio
# Telepítjük a tinycudann-t is, ami a gyors tanításhoz elengedhetetlen
!pip install ninja git+https://github.com/NVlabs/tiny-cuda-nn/#subdirectory=bindings/torch

### 2. Adatok kicsomagolása

In [ ]:
!rm -rf /content/ScanData
!mkdir /content/ScanData
!unzip -q /content/Scan.zip -d /content/ScanData/
print("✅ Adatok sikeresen kicsomagolva a /content/ScanData mappába!")

### 3. ARKit adatok konvertálása NerfStudio formátumra
Az Apple LiDAR és ARKit egyedi formátumban menti a 3D mátrixokat. Ez a script átalakítja OpenCV szabványra, amit a Gaussian Splatting motor megért.

In [ ]:
import json
import os
from PIL import Image

def convert_arkit_to_nerfstudio(data_dir):
    input_json_path = os.path.join(data_dir, "transforms.json")
    output_json_path = os.path.join(data_dir, "transforms_nerfstudio.json")
    
    if not os.path.exists(input_json_path):
        # Hátha egy mappával beljebb van a zip miatt
        subdirs = [os.path.join(data_dir, d) for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]
        if subdirs and os.path.exists(os.path.join(subdirs[0], "transforms.json")):
            data_dir = subdirs[0]
            input_json_path = os.path.join(data_dir, "transforms.json")
            output_json_path = os.path.join(data_dir, "transforms_nerfstudio.json")
        else:
            print(f"❌ HIBA: Nem találom a transforms.json fájlt itt: {input_json_path}")
            return data_dir
            
    with open(input_json_path, 'r') as f:
        data = json.load(f)
        
    frames = data.get("frames", [])
    if not frames:
        print("❌ HIBA: Nincsenek képkockák a transforms.json-ben!")
        return data_dir
        
    # Kép felbontásának beolvasása az első képből
    first_image_path = os.path.join(data_dir, frames[0]["file_path"])
    if not os.path.exists(first_image_path):
        print(f"❌ HIBA: Nem találom az első képet: {first_image_path}.")
        return data_dir
        
    with Image.open(first_image_path) as img:
        w, h = img.size
        
    intrinsics = frames[0]["intrinsics_matrix"]
    fl_x = intrinsics[0][0]
    fl_y = intrinsics[1][1]
    cx = intrinsics[2][0]
    cy = intrinsics[2][1]
    
    out_data = {
        "camera_model": "OPENCV",
        "orientation_override": "none",
        "fl_x": fl_x,
        "fl_y": fl_y,
        "cx": cx,
        "cy": cy,
        "w": w,
        "h": h,
        "frames": []
    }
    
    for frame in frames:
        matrix = frame["transform_matrix"]
        # ARKit (Right, Up, Back) -> OpenCV (Right, Down, Forward)
        c2w = [
            [matrix[0][0], -matrix[0][1], -matrix[0][2], matrix[0][3]],
            [matrix[1][0], -matrix[1][1], -matrix[1][2], matrix[1][3]],
            [matrix[2][0], -matrix[2][1], -matrix[2][2], matrix[2][3]],
            [matrix[3][0], -matrix[3][1], -matrix[3][2], matrix[3][3]]
        ]
        
        out_frame = {
            "file_path": frame["file_path"],
            "transform_matrix": c2w
        }
        
        if "depth_path" in frame:
            out_frame["depth_file_path"] = frame["depth_path"]
            
        out_data["frames"].append(out_frame)
        
    with open(output_json_path, 'w') as f:
        json.dump(out_data, f, indent=4)
        
    print(f"✅ Sikeresen konvertáltam {len(frames)} képkockát NerfStudio formátumra!")
    return data_dir

actual_data_dir = convert_arkit_to_nerfstudio("/content/ScanData")
with open("/content/actual_data_dir.txt", "w") as f:
    f.write(actual_data_dir)

### 4. Tanítás (Gaussian Splatting generálás)
Elindítjuk a `splatfacto` AI modellt. Ez kb. 10-15 percet vesz igénybe a feltöltött képek számától függően.

In [ ]:
!DATA_DIR=$(cat /content/actual_data_dir.txt) && ns-train splatfacto --data $DATA_DIR --pipeline.model.cull-alpha-thresh 0.005 --pipeline.model.continue-cull-post-densification False --pipeline.model.sh-degree 3 --max-num-iterations 7000 --timestamp "mobilscan_run"

### 5. Modell exportálása és letöltése
A kigenerált modellt kimentjük szabványos `.ply` formátumba.

In [ ]:
!ns-export gaussian-splat --load-config outputs/mobilscan_run/splatfacto/mobilscan_run/config.yml --output-dir /content/export
print("✅ Modell exportálva! Keresd a bal oldali Fájlok (Files) menüben a /content/export/splat.ply fájlt, kattints rá jobb gombbal és 'Download'!")
print("Ezután megnyithatod bármilyen Gaussian Splatting nézegetőben, pl: https://playcanvas.com/supersplat/editor")